# WiDS 2026 Wildfire | Breakthrough Survival Stack + Native XGB Survival + Consensus Blend


In [1]:

import os, sys, json, math, warnings, subprocess, re, hashlib
warnings.filterwarnings("ignore")

RUN_MODE = "full"   # "full" or "fast"
DO_OOF = True
AUTO_INSTALL_EXTRA = False

WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else ("/mnt/data" if os.path.isdir("/mnt/data") else ".")
OUTPUT_PATH = os.path.join(WORK_DIR, "submission.csv")

def find_data_dir():
    candidates = [
        "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/widsworldwide-globaldathon26",
        "/kaggle/input/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/wiDSworldwide_globaldathon26",
        "/kaggle/input",
        ".",
        "/mnt/data",
    ]
    for path in candidates:
        if all(os.path.exists(os.path.join(path, f)) for f in ["train.csv", "test.csv", "sample_submission.csv"]):
            return path
    if os.path.isdir("/kaggle/input"):
        for root, dirs, files in os.walk("/kaggle/input"):
            fs = set(files)
            if {"train.csv", "test.csv", "sample_submission.csv"}.issubset(fs):
                return root
    raise FileNotFoundError("Could not find competition data directory.")

DATA_DIR = find_data_dir()

def ensure_pkg(pkg, import_name=None, allow_install=False):
    name = import_name or pkg
    try:
        __import__(name)
        return True
    except Exception:
        if not allow_install:
            return False
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
            __import__(name)
            return True
        except Exception as e:
            print(f"[WARN] package unavailable: {pkg} -> {e}")
            return False

ensure_pkg("lightgbm", "lightgbm", AUTO_INSTALL_EXTRA)
ensure_pkg("catboost", "catboost", AUTO_INSTALL_EXTRA)
HAS_XGB = ensure_pkg("xgboost", "xgboost", AUTO_INSTALL_EXTRA)

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import erf, expit
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression

import lightgbm as lgb
from catboost import CatBoostClassifier
if HAS_XGB:
    import xgboost as xgb

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

print("DATA_DIR =", DATA_DIR)
print("TRAIN =", train_df.shape, "TEST =", test_df.shape, "HAS_XGB =", HAS_XGB)

def create_features(df, fit_df=None):
    ref = fit_df if fit_df is not None else df
    r = df.copy()

    dist = r["dist_min_ci_0_5h"].clip(lower=1)
    speed = r["closing_speed_m_per_h"]
    perims = r["num_perimeters_0_5h"]
    area_first = r["area_first_ha"]
    area_growth = r["area_growth_rate_ha_per_h"]
    radial = r["radial_growth_rate_m_per_h"].clip(lower=0)
    align = r["alignment_abs"]
    fire_radius = np.sqrt(np.maximum(area_first, 0) * 10000.0 / np.pi)
    closing_pos = speed.clip(lower=0)
    eff_close = closing_pos + radial

    r["dist_km"] = dist / 1000.0
    r["log_distance"] = np.log1p(dist)
    r["inv_distance"] = 1.0 / (dist / 1000.0 + 0.1)
    r["inv_distance_sq"] = r["inv_distance"] ** 2
    r["sqrt_distance"] = np.sqrt(dist)
    r["dist_km_sq"] = r["dist_km"] ** 2
    r["dist_km_cb"] = r["dist_km"] ** 3

    r["fire_radius_km"] = fire_radius / 1000.0
    r["radius_to_dist"] = fire_radius / dist
    r["area_to_dist_ratio"] = area_first / (dist / 1000.0 + 0.1)
    r["log_area_dist_ratio"] = np.log1p(area_first) - np.log1p(dist)

    r["has_movement"] = (perims > 1).astype(float)
    r["effective_closing_speed"] = eff_close
    r["eta_hours"] = np.where(closing_pos > 0.01, dist / closing_pos, 9999.0).clip(max=9999.0)
    r["eta_effective"] = np.where(eff_close > 0.01, dist / eff_close, 9999.0).clip(max=9999.0)
    r["log_eta"] = np.log1p(r["eta_hours"])
    r["log_eta_effective"] = np.log1p(r["eta_effective"])

    r["threat_score"] = align * speed / np.log1p(dist)
    r["threat_score_eff"] = align * eff_close / np.log1p(dist)
    r["threat_score_sq"] = r["threat_score_eff"] ** 2
    r["fire_urgency"] = perims * speed
    r["growth_intensity"] = area_growth * perims
    r["speed_alignment"] = speed * align
    r["effective_alignment"] = eff_close * align

    r["projected_advance_pos"] = np.maximum(r["projected_advance_m"], 0)
    r["closing_distance_pos"] = np.maximum(-r["dist_change_ci_0_5h"], 0)
    r["slope_toward"] = np.maximum(-r["dist_slope_ci_0_5h"], 0)
    r["accel_toward"] = np.maximum(-r["dist_accel_m_per_h2"], 0)

    r["zone_3km"] = (dist < 3000).astype(float)
    r["zone_near"] = (dist < 5000).astype(float)
    r["zone_warning"] = ((dist >= 5000) & (dist < 10000)).astype(float)
    r["zone_far"] = (dist >= 10000).astype(float)
    r["zone_20km"] = (dist < 20000).astype(float)

    r["is_summer"] = r["event_start_month"].isin([6, 7, 8]).astype(float)
    r["is_afternoon"] = ((r["event_start_hour"] >= 12) & (r["event_start_hour"] < 20)).astype(float)
    r["is_night"] = ((r["event_start_hour"] <= 6) | (r["event_start_hour"] >= 21)).astype(float)

    r["hour_sin"] = np.sin(2 * np.pi * r["event_start_hour"] / 24.0)
    r["hour_cos"] = np.cos(2 * np.pi * r["event_start_hour"] / 24.0)
    r["month_sin"] = np.sin(2 * np.pi * (r["event_start_month"] - 1) / 12.0)
    r["month_cos"] = np.cos(2 * np.pi * (r["event_start_month"] - 1) / 12.0)
    r["dow_sin"] = np.sin(2 * np.pi * r["event_start_dayofweek"] / 7.0)
    r["dow_cos"] = np.cos(2 * np.pi * r["event_start_dayofweek"] / 7.0)

    def pct_rank(values, ref_values):
        ref_values = np.asarray(ref_values, dtype=float)
        if ref_values.size == 0:
            return np.zeros(len(values), dtype=float)
        ref_sorted = np.sort(ref_values)
        return np.searchsorted(ref_sorted, np.asarray(values, dtype=float), side="right") / ref_sorted.size

    ref_dist = ref["dist_min_ci_0_5h"].clip(lower=1).values
    ref_eff = (
        ref["closing_speed_m_per_h"].clip(lower=0).values +
        ref["radial_growth_rate_m_per_h"].clip(lower=0).values
    )
    ref_threat = ref["alignment_abs"].values * ref_eff / np.log1p(ref["dist_min_ci_0_5h"].clip(lower=1).values)

    r["dist_rank_global"] = pct_rank(dist.values, ref_dist)
    r["eff_rank_global"] = pct_rank(eff_close.values, ref_eff)
    r["threat_rank_global"] = pct_rank(r["threat_score_eff"].values, ref_threat)

    ref_near = ref["dist_min_ci_0_5h"].clip(lower=1).values < 5000
    cur_near = dist.values < 5000
    near_speed_rank = np.zeros(len(r), dtype=float)
    far_threat_rank = np.zeros(len(r), dtype=float)
    near_ref_eff = ref_eff[ref_near]
    far_ref_threat = ref_threat[~ref_near]
    if cur_near.any():
        near_speed_rank[cur_near] = pct_rank(eff_close.values[cur_near], near_ref_eff)
    if (~cur_near).any():
        far_threat_rank[~cur_near] = pct_rank(r["threat_score_eff"].values[~cur_near], far_ref_threat)
    r["near_speed_rank"] = near_speed_rank
    r["far_threat_rank"] = far_threat_rank

    drop_cols = [
        "relative_growth_0_5h",
        "projected_advance_m",
        "centroid_displacement_m",
        "centroid_speed_m_per_h",
        "closing_speed_abs_m_per_h",
        "area_growth_abs_0_5h",
    ]
    r = r.drop(columns=[c for c in drop_cols if c in r.columns])
    r = r.replace([np.inf, -np.inf], np.nan).fillna(0)
    return r

train_proc = create_features(train_df, fit_df=train_df)
test_proc = create_features(test_df, fit_df=train_df)

time_values = train_df["time_to_hit_hours"].values
event_values = train_df["event"].astype(int).values
dist_train = train_df["dist_min_ci_0_5h"].values
dist_test = test_df["dist_min_ci_0_5h"].values

row_rel_train = np.clip(
    0.30
    + 0.20 * train_proc["has_movement"].values
    + 0.20 * (1.0 - train_proc["low_temporal_resolution_0_5h"].values)
    + 0.15 * np.clip(train_proc.get("dist_fit_r2_0_5h", pd.Series(0.0, index=train_proc.index)).values, 0.0, 1.0)
    + 0.15 * np.clip(train_proc["num_perimeters_0_5h"].values / 5.0, 0.0, 1.0),
    0.35, 0.97
)
row_rel_test = np.clip(
    0.30
    + 0.20 * test_proc["has_movement"].values
    + 0.20 * (1.0 - test_proc["low_temporal_resolution_0_5h"].values)
    + 0.15 * np.clip(test_proc.get("dist_fit_r2_0_5h", pd.Series(0.0, index=test_proc.index)).values, 0.0, 1.0)
    + 0.15 * np.clip(test_proc["num_perimeters_0_5h"].values / 5.0, 0.0, 1.0),
    0.35, 0.97
)

def compute_c_index(time, event, risk):
    n = len(time)
    conc = 0.0
    comp = 0.0
    for i in range(n):
        if event[i] != 1:
            continue
        for j in range(n):
            if i == j or time[i] >= time[j]:
                continue
            comp += 1
            if risk[i] > risk[j]:
                conc += 1
            elif risk[i] == risk[j]:
                conc += 0.5
    return conc / comp if comp else 0.5

def compute_brier(time, event, prob, horizon):
    valid = ~((event == 0) & (time < horizon))
    if valid.sum() == 0:
        return 0.25
    y_true = ((event == 1) & (time <= horizon)).astype(float)[valid]
    prob = np.clip(prob[valid], 0, 1)
    return float(np.mean((prob - y_true) ** 2))

def compute_hybrid_score(time, event, p24, p48, p72):
    risk = 0.3 * p24 + 0.4 * p48 + 0.3 * p72
    c_idx = compute_c_index(time, event, risk)
    b24 = compute_brier(time, event, p24, 24)
    b48 = compute_brier(time, event, p48, 48)
    b72 = compute_brier(time, event, p72, 72)
    wb = 0.3 * b24 + 0.4 * b48 + 0.3 * b72
    return 0.3 * c_idx + 0.7 * (1.0 - wb), c_idx, wb, b24, b48, b72

def enforce_monotonicity(preds):
    out = np.clip(preds, 0, 1)
    for i in range(1, out.shape[1]):
        out[:, i] = np.maximum(out[:, i], out[:, i - 1])
    return out

def make_binary_target(time_vals, event_vals, horizon):
    unknown = (event_vals == 0) & (time_vals < horizon)
    y = ((event_vals == 1) & (time_vals <= horizon)).astype(int)
    return y, ~unknown

def compute_ipcw_weights(times, events, horizon):
    times = np.asarray(times, dtype=float)
    events = np.asarray(events, dtype=int)
    unique_t = np.sort(np.unique(times))
    surv = np.ones(len(unique_t), dtype=float)
    for i, t in enumerate(unique_t):
        at_risk = np.sum(times >= t)
        cens = np.sum((times == t) & (events == 0))
        if at_risk > 0:
            surv[i] = 1.0 - cens / at_risk
        if i > 0:
            surv[i] *= surv[i - 1]
    def G(t):
        idx = np.searchsorted(unique_t, t, side="right") - 1
        if idx >= 0:
            return max(float(surv[idx]), 0.01)
        return 1.0
    weights = np.ones(len(times), dtype=float)
    for i in range(len(times)):
        if events[i] == 1 and times[i] <= horizon:
            weights[i] = 1.0 / G(times[i])
        elif times[i] >= horizon:
            weights[i] = 1.0 / G(horizon)
        else:
            weights[i] = 0.0
    return weights

def fit_with_optional_weight(model, X, y, sample_weight=None):
    if sample_weight is None:
        model.fit(X, y)
        return model
    if isinstance(model, Pipeline):
        last_name = model.steps[-1][0]
        try:
            model.fit(X, y, **{f"{last_name}__sample_weight": sample_weight})
            return model
        except Exception:
            model.fit(X, y)
            return model
    try:
        model.fit(X, y, sample_weight=sample_weight)
        return model
    except Exception:
        model.fit(X, y)
        return model

def safe_predict_proba(model, X):
    p = model.predict_proba(X)
    if isinstance(p, list):
        p = np.asarray(p)
    return p[:, 1] if p.ndim == 2 else p

def soft_gate(dist_m, center, width):
    z = np.clip((dist_m - center) / max(width, 1.0), -60, 60)
    return 1.0 / (1.0 + np.exp(z))

def stabilize(pred, anchor, alpha):
    return np.clip(alpha * pred + (1.0 - alpha) * anchor, 0, 1)

def stabilize_rowwise(pred, anchor, alpha):
    alpha = np.asarray(alpha, dtype=float)
    return np.clip(alpha * pred + (1.0 - alpha) * anchor, 0, 1)

def repeated_isotonic(train_signal, test_signal, y, mask, seeds, n_splits=5):
    train_signal = np.asarray(train_signal, dtype=float)
    test_signal = np.asarray(test_signal, dtype=float)
    valid_idx = np.where(mask)[0]
    yv = y[valid_idx]
    oof = np.zeros(len(train_signal), dtype=float)
    cnt = np.zeros(len(train_signal), dtype=float)
    full_fill = np.zeros(len(train_signal), dtype=float)
    test_pred = np.zeros(len(test_signal), dtype=float)

    for seed in seeds:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr_sub, va_sub in cv.split(valid_idx, yv):
            tr_idx = valid_idx[tr_sub]
            va_idx = valid_idx[va_sub]
            ir = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip")
            ir.fit(train_signal[tr_idx], y[tr_idx])
            oof[va_idx] += ir.predict(train_signal[va_idx])
            cnt[va_idx] += 1.0
        ir_full = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip")
        ir_full.fit(train_signal[valid_idx], yv)
        full_fill += ir_full.predict(train_signal)
        test_pred += ir_full.predict(test_signal)

    full_fill /= len(seeds)
    test_pred /= len(seeds)
    nz = cnt > 0
    oof[nz] /= cnt[nz]
    oof[~nz] = full_fill[~nz]
    return np.clip(oof, 0, 1), np.clip(test_pred, 0, 1)

def repeated_binary_model(X_train, X_test, y, mask, build_model, seeds, n_splits=5, horizon=None, use_ipcw=False):
    n_train = len(X_train)
    n_test = len(X_test)
    valid_idx = np.where(mask)[0]
    Xv = X_train.iloc[valid_idx]
    yv = y[valid_idx]
    oof = np.zeros(n_train, dtype=float)
    cnt = np.zeros(n_train, dtype=float)
    full_fill = np.zeros(n_train, dtype=float)
    test_pred = np.zeros(n_test, dtype=float)

    for seed in seeds:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr_sub, va_sub in cv.split(Xv, yv):
            tr_idx = valid_idx[tr_sub]
            va_idx = valid_idx[va_sub]
            model = build_model(seed)
            sw = None
            if use_ipcw and horizon is not None:
                sw = compute_ipcw_weights(time_values[tr_idx], event_values[tr_idx], horizon)
            fit_with_optional_weight(model, X_train.iloc[tr_idx], y[tr_idx], sw)
            oof[va_idx] += safe_predict_proba(model, X_train.iloc[va_idx])
            cnt[va_idx] += 1.0

        model_full = build_model(seed)
        sw_full = None
        if use_ipcw and horizon is not None:
            sw_full = compute_ipcw_weights(time_values[valid_idx], event_values[valid_idx], horizon)
        fit_with_optional_weight(model_full, Xv, yv, sw_full)
        full_fill += safe_predict_proba(model_full, X_train)
        test_pred += safe_predict_proba(model_full, X_test)

    full_fill /= len(seeds)
    test_pred /= len(seeds)
    nz = cnt > 0
    oof[nz] /= cnt[nz]
    oof[~nz] = full_fill[~nz]
    return np.clip(oof, 0, 1), np.clip(test_pred, 0, 1)

def aft_event_prob(pred, horizon, dist_name, scale):
    z = (np.log(float(horizon)) - np.asarray(pred, dtype=float)) / max(float(scale), 1e-6)
    dist_name = str(dist_name).lower()
    if dist_name == "normal":
        out = 0.5 * (1.0 + erf(z / np.sqrt(2.0)))
    elif dist_name == "logistic":
        out = expit(z)
    else:
        out = 1.0 - np.exp(-np.exp(np.clip(z, -60.0, 60.0)))
    return np.clip(out, 0.0, 1.0)

def aft_pred_to_probs(pred, horizons, dist_name, scale):
    return np.column_stack([aft_event_prob(pred, h, dist_name, scale) for h in horizons])

def repeated_xgb_aft_models(X_train, X_test, lower, upper, configs, seeds, n_splits=5):
    horizons = [12, 24, 48]
    n_train = len(X_train)
    n_test = len(X_test)
    oof = np.zeros((n_train, len(horizons)), dtype=float)
    cnt = np.zeros(n_train, dtype=float)
    full_fill = np.zeros((n_train, len(horizons)), dtype=float)
    test_pred = np.zeros((n_test, len(horizons)), dtype=float)
    feature_names = list(X_train.columns)
    dtest = xgb.DMatrix(X_test.to_numpy(dtype=float), feature_names=feature_names)
    full_models = 0

    for cfg in configs:
        cfg = dict(cfg)
        dist_name = cfg.get("aft_loss_distribution", "normal")
        scale = float(cfg.get("aft_loss_distribution_scale", 1.0))
        num_boost_round = int(cfg.pop("num_boost_round", 500))

        for seed in seeds:
            cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

            for tr_idx, va_idx in cv.split(X_train, event_values):
                try:
                    dtr = xgb.DMatrix(X_train.iloc[tr_idx].to_numpy(dtype=float), feature_names=feature_names)
                    dva = xgb.DMatrix(X_train.iloc[va_idx].to_numpy(dtype=float), feature_names=feature_names)
                    dtr.set_float_info("label_lower_bound", np.asarray(lower[tr_idx], dtype=float))
                    dtr.set_float_info("label_upper_bound", np.asarray(upper[tr_idx], dtype=float))
                    params = dict(cfg)
                    params.update({
                        "objective": "survival:aft",
                        "eval_metric": "aft-nloglik",
                        "tree_method": "hist",
                        "verbosity": 0,
                        "seed": seed,
                        "nthread": max(1, int(os.cpu_count() or 4)),
                        "aft_loss_distribution": dist_name,
                        "aft_loss_distribution_scale": scale,
                    })
                    booster = xgb.train(params, dtr, num_boost_round=num_boost_round, verbose_eval=False)
                    pred_va = booster.predict(dva)
                    oof[va_idx] += aft_pred_to_probs(pred_va, horizons, dist_name, scale)
                    cnt[va_idx] += 1.0
                except Exception:
                    oof[va_idx] += 0.5
                    cnt[va_idx] += 1.0

            try:
                dfull = xgb.DMatrix(X_train.to_numpy(dtype=float), feature_names=feature_names)
                dfull.set_float_info("label_lower_bound", np.asarray(lower, dtype=float))
                dfull.set_float_info("label_upper_bound", np.asarray(upper, dtype=float))
                params = dict(cfg)
                params.update({
                    "objective": "survival:aft",
                    "eval_metric": "aft-nloglik",
                    "tree_method": "hist",
                    "verbosity": 0,
                    "seed": seed,
                    "nthread": max(1, int(os.cpu_count() or 4)),
                    "aft_loss_distribution": dist_name,
                    "aft_loss_distribution_scale": scale,
                })
                booster_full = xgb.train(params, dfull, num_boost_round=num_boost_round, verbose_eval=False)
                full_fill += aft_pred_to_probs(booster_full.predict(dfull), horizons, dist_name, scale)
                test_pred += aft_pred_to_probs(booster_full.predict(dtest), horizons, dist_name, scale)
                full_models += 1
            except Exception:
                full_fill += 0.5
                test_pred += 0.5
                full_models += 1

    if full_models > 0:
        full_fill /= full_models
        test_pred /= full_models
    else:
        full_fill[:] = 0.5
        test_pred[:] = 0.5

    nz = cnt > 0
    oof[nz] /= cnt[nz, None]
    oof[~nz] = full_fill[~nz]
    return np.clip(oof, 0.0, 1.0), np.clip(test_pred, 0.0, 1.0)

def repeated_xgb_cox_signal(X_train, X_test, signed_time, configs, seeds, n_splits=5):
    n_train = len(X_train)
    n_test = len(X_test)
    oof = np.zeros(n_train, dtype=float)
    cnt = np.zeros(n_train, dtype=float)
    full_fill = np.zeros(n_train, dtype=float)
    test_pred = np.zeros(n_test, dtype=float)
    feature_names = list(X_train.columns)
    y_signed = np.asarray(signed_time, dtype=float)
    dtest = xgb.DMatrix(X_test.to_numpy(dtype=float), feature_names=feature_names)
    full_models = 0

    for cfg in configs:
        cfg = dict(cfg)
        num_boost_round = int(cfg.pop("num_boost_round", 500))

        for seed in seeds:
            cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

            for tr_idx, va_idx in cv.split(X_train, event_values):
                try:
                    dtr = xgb.DMatrix(
                        X_train.iloc[tr_idx].to_numpy(dtype=float),
                        label=y_signed[tr_idx],
                        feature_names=feature_names,
                    )
                    dva = xgb.DMatrix(X_train.iloc[va_idx].to_numpy(dtype=float), feature_names=feature_names)
                    params = dict(cfg)
                    params.update({
                        "objective": "survival:cox",
                        "eval_metric": "cox-nloglik",
                        "tree_method": "hist",
                        "verbosity": 0,
                        "seed": seed,
                        "nthread": max(1, int(os.cpu_count() or 4)),
                    })
                    booster = xgb.train(params, dtr, num_boost_round=num_boost_round, verbose_eval=False)
                    pred_va = np.log(np.clip(booster.predict(dva), 1e-12, None))
                    oof[va_idx] += pred_va
                    cnt[va_idx] += 1.0
                except Exception:
                    oof[va_idx] += 0.0
                    cnt[va_idx] += 1.0

            try:
                dfull = xgb.DMatrix(
                    X_train.to_numpy(dtype=float),
                    label=y_signed,
                    feature_names=feature_names,
                )
                params = dict(cfg)
                params.update({
                    "objective": "survival:cox",
                    "eval_metric": "cox-nloglik",
                    "tree_method": "hist",
                    "verbosity": 0,
                    "seed": seed,
                    "nthread": max(1, int(os.cpu_count() or 4)),
                })
                booster_full = xgb.train(params, dfull, num_boost_round=num_boost_round, verbose_eval=False)
                full_fill += np.log(np.clip(booster_full.predict(dfull), 1e-12, None))
                test_pred += np.log(np.clip(booster_full.predict(dtest), 1e-12, None))
                full_models += 1
            except Exception:
                full_fill += 0.0
                test_pred += 0.0
                full_models += 1

    if full_models > 0:
        full_fill /= full_models
        test_pred /= full_models
    else:
        full_fill[:] = 0.0
        test_pred[:] = 0.0

    nz = cnt > 0
    oof[nz] /= cnt[nz]
    oof[~nz] = full_fill[~nz]
    return oof, test_pred

FAST_SEEDS = (123, 456, 789)
BASE_SEEDS = (123, 456, 789, 777, 666, 1511, 2025, 2026)
TREE_SEEDS_FULL = BASE_SEEDS + (239, 279, 70, 77)
CAT_SEEDS_FULL = BASE_SEEDS[:6]

def choose_seeds(full_seeds):
    return FAST_SEEDS if RUN_MODE == "fast" else tuple(full_seeds)

RAW_FEATURES = [c for c in train_df.columns if c not in ["event_id", "event", "time_to_hit_hours"]]

COX_FEATURES = [
    "dist_km", "log_distance", "inv_distance",
    "closing_speed_m_per_h", "radial_growth_rate_m_per_h",
    "alignment_abs", "threat_score_eff", "log_eta_effective", "eta_effective",
    "area_to_dist_ratio", "fire_radius_km",
    "num_perimeters_0_5h", "has_movement",
    "near_speed_rank", "far_threat_rank",
    "low_temporal_resolution_0_5h", "dt_first_last_0_5h",
    "is_summer", "is_afternoon", "hour_sin", "hour_cos",
    "zone_near", "zone_far",
]
COX_FEATURES = [c for c in COX_FEATURES if c in train_proc.columns]

NEAR_LGB_FEATURES = [
    "closing_speed_m_per_h", "radial_growth_rate_m_per_h",
    "alignment_abs", "num_perimeters_0_5h", "area_growth_rate_ha_per_h",
    "eta_effective", "log_eta_effective", "dist_km", "threat_score_eff",
    "near_speed_rank", "event_start_hour", "is_afternoon", "fire_urgency",
    "area_first_ha", "fire_radius_km", "low_temporal_resolution_0_5h",
    "dt_first_last_0_5h",
]
NEAR_LGB_FEATURES = [c for c in NEAR_LGB_FEATURES if c in train_proc.columns]

FAR_LGB_FEATURES = [
    "dist_km", "log_distance", "inv_distance",
    "closing_speed_m_per_h", "alignment_abs",
    "threat_score_eff", "log_eta_effective", "eta_effective",
    "area_to_dist_ratio", "num_perimeters_0_5h",
    "far_threat_rank", "is_summer", "zone_far",
    "low_temporal_resolution_0_5h", "dt_first_last_0_5h",
    "log1p_growth", "dist_fit_r2_0_5h",
]
FAR_LGB_FEATURES = [c for c in FAR_LGB_FEATURES if c in train_proc.columns]

GLOBAL_TREE_FEATURES = sorted(set(
    NEAR_LGB_FEATURES + FAR_LGB_FEATURES + [
        "inv_distance_sq", "sqrt_distance", "threat_score", "growth_intensity",
        "speed_alignment", "effective_alignment", "zone_warning",
        "hour_sin", "hour_cos", "month_sin", "month_cos", "dow_sin", "dow_cos",
        "projected_advance_pos", "closing_distance_pos", "slope_toward",
        "log_area_dist_ratio", "fire_radius_km", "log1p_area_first",
        "dist_rank_global", "eff_rank_global", "threat_rank_global",
    ]
))
GLOBAL_TREE_FEATURES = [c for c in GLOBAL_TREE_FEATURES if c in train_proc.columns]

LINEAR_FEATURES = [
    "dist_km", "log_distance", "inv_distance",
    "closing_speed_m_per_h", "radial_growth_rate_m_per_h",
    "alignment_abs", "threat_score_eff", "log_eta_effective", "eta_effective",
    "area_to_dist_ratio", "fire_radius_km",
    "num_perimeters_0_5h", "has_movement",
    "near_speed_rank", "far_threat_rank",
    "low_temporal_resolution_0_5h", "dt_first_last_0_5h",
    "is_summer", "is_afternoon", "hour_sin", "hour_cos",
    "zone_near", "zone_far",
]
LINEAR_FEATURES = [c for c in LINEAR_FEATURES if c in train_proc.columns]

X_near_lgb_train = train_proc[NEAR_LGB_FEATURES].copy()
X_near_lgb_test = test_proc[NEAR_LGB_FEATURES].copy()
X_far_lgb_train = train_proc[FAR_LGB_FEATURES].copy()
X_far_lgb_test = test_proc[FAR_LGB_FEATURES].copy()
X_global_train = train_proc[GLOBAL_TREE_FEATURES].copy()
X_global_test = test_proc[GLOBAL_TREE_FEATURES].copy()
X_linear_train = train_proc[LINEAR_FEATURES].copy()
X_linear_test = test_proc[LINEAR_FEATURES].copy()

X_xgb_surv_train = train_proc[sorted(set(GLOBAL_TREE_FEATURES + LINEAR_FEATURES + COX_FEATURES))].copy()
X_xgb_surv_test = test_proc[X_xgb_surv_train.columns].copy()

def make_lgb_builder(params):
    def _build(seed):
        p = dict(params)
        p.update(dict(objective="binary", random_state=seed, n_jobs=-1, verbose=-1))
        return lgb.LGBMClassifier(**p)
    return _build

def make_cat_builder(params):
    def _build(seed):
        p = dict(params)
        p.update(dict(
            loss_function="Logloss",
            eval_metric="Logloss",
            random_seed=seed,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1,
        ))
        return CatBoostClassifier(**p)
    return _build

def make_logit_builder(C):
    def _build(seed):
        return Pipeline([
            ("scaler", StandardScaler()),
            ("lr", LogisticRegression(C=C, penalty="l2", solver="lbfgs", max_iter=4000)),
        ])
    return _build

global_lgb_cfgs = {
    12: {"max_depth": 2, "learning_rate": 0.040, "n_estimators": 220, "subsample": 0.80, "colsample_bytree": 0.80,
         "min_child_samples": 4, "reg_alpha": 0.5, "reg_lambda": 1.5, "num_leaves": 4},
    24: {"max_depth": 2, "learning_rate": 0.035, "n_estimators": 260, "subsample": 0.80, "colsample_bytree": 0.80,
         "min_child_samples": 6, "reg_alpha": 0.8, "reg_lambda": 2.5, "num_leaves": 4},
    48: {"max_depth": 3, "learning_rate": 0.030, "n_estimators": 260, "subsample": 0.85, "colsample_bytree": 0.85,
         "min_child_samples": 8, "reg_alpha": 1.0, "reg_lambda": 3.0, "num_leaves": 6},
}

global_cat_cfgs = {
    12: {"iterations": 300, "learning_rate": 0.030, "depth": 4, "l2_leaf_reg": 10.0,
         "bootstrap_type": "Bernoulli", "subsample": 0.85, "random_strength": 0.8},
    24: {"iterations": 320, "learning_rate": 0.030, "depth": 4, "l2_leaf_reg": 12.0,
         "bootstrap_type": "Bernoulli", "subsample": 0.85, "random_strength": 0.8},
    48: {"iterations": 340, "learning_rate": 0.028, "depth": 4, "l2_leaf_reg": 14.0,
         "bootstrap_type": "Bernoulli", "subsample": 0.85, "random_strength": 0.8},
}

near_lgb_cfgs = {
    12: {"max_depth": 2, "learning_rate": 0.040, "n_estimators": 220, "subsample": 0.80, "colsample_bytree": 0.80,
         "min_child_samples": 4, "reg_alpha": 0.5, "reg_lambda": 1.5, "num_leaves": 4},
    24: {"max_depth": 2, "learning_rate": 0.040, "n_estimators": 200, "subsample": 0.80, "colsample_bytree": 0.80,
         "min_child_samples": 4, "reg_alpha": 0.5, "reg_lambda": 1.5, "num_leaves": 4},
    48: {"max_depth": 2, "learning_rate": 0.050, "n_estimators": 150, "subsample": 0.80, "colsample_bytree": 0.80,
         "min_child_samples": 3, "reg_alpha": 0.3, "reg_lambda": 1.0, "num_leaves": 4},
}

far_lgb_cfgs = {
    24: {"max_depth": 2, "learning_rate": 0.030, "n_estimators": 200, "subsample": 0.70, "colsample_bytree": 0.70,
         "min_child_samples": 8, "reg_alpha": 1.0, "reg_lambda": 3.0, "num_leaves": 4},
    48: {"max_depth": 2, "learning_rate": 0.050, "n_estimators": 150, "subsample": 0.80, "colsample_bytree": 0.80,
         "min_child_samples": 6, "reg_alpha": 0.5, "reg_lambda": 2.0, "num_leaves": 4},
}

linear_cs = {12: 0.60, 24: 0.80, 48: 1.00}

XGB_AFT_CFGS = [
    {"max_depth": 3, "eta": 0.030, "subsample": 0.85, "colsample_bytree": 0.85, "min_child_weight": 5, "lambda": 2.0, "alpha": 0.5, "num_boost_round": 500, "aft_loss_distribution": "normal", "aft_loss_distribution_scale": 1.00},
    {"max_depth": 2, "eta": 0.040, "subsample": 0.80, "colsample_bytree": 0.80, "min_child_weight": 4, "lambda": 2.5, "alpha": 0.8, "num_boost_round": 420, "aft_loss_distribution": "logistic", "aft_loss_distribution_scale": 1.15},
    {"max_depth": 3, "eta": 0.025, "subsample": 0.90, "colsample_bytree": 0.80, "min_child_weight": 6, "lambda": 3.0, "alpha": 1.0, "num_boost_round": 600, "aft_loss_distribution": "extreme", "aft_loss_distribution_scale": 1.00},
]
XGB_COX_CFGS = [
    {"max_depth": 2, "eta": 0.040, "subsample": 0.85, "colsample_bytree": 0.80, "min_child_weight": 5, "lambda": 2.0, "alpha": 0.5, "num_boost_round": 420},
    {"max_depth": 3, "eta": 0.030, "subsample": 0.80, "colsample_bytree": 0.85, "min_child_weight": 6, "lambda": 3.0, "alpha": 1.0, "num_boost_round": 520},
]

if RUN_MODE == "fast":
    XGB_AFT_CFGS = XGB_AFT_CFGS[:2]
    XGB_COX_CFGS = XGB_COX_CFGS[:1]

ANCHOR_SEEDS = choose_seeds(BASE_SEEDS[:6])
TREE_SEEDS = choose_seeds(TREE_SEEDS_FULL)
CAT_SEEDS = choose_seeds(CAT_SEEDS_FULL)
LINEAR_SEEDS = choose_seeds(BASE_SEEDS[:6])
XGB_SURV_SEEDS = choose_seeds(BASE_SEEDS[:6])

y_map = {}
mask_map = {}
for h in [12, 24, 48]:
    y_map[h], mask_map[h] = make_binary_target(time_values, event_values, h)

anchor_signal_train = 1.0 / (train_proc["dist_km"].values + 0.25)
anchor_signal_test = 1.0 / (test_proc["dist_km"].values + 0.25)

anchor_oof, anchor_test = {}, {}
for h in [12, 24, 48]:
    anchor_oof[h], anchor_test[h] = repeated_isotonic(
        anchor_signal_train, anchor_signal_test, y_map[h], mask_map[h], ANCHOR_SEEDS, n_splits=5
    )
    print(f"anchor@{h}      Brier = {compute_brier(time_values, event_values, anchor_oof[h], h):.6f}")

global_lgb_oof, global_lgb_test = {}, {}
global_cat_oof, global_cat_test = {}, {}
linear_oof, linear_test = {}, {}

for h in [12, 24, 48]:
    global_lgb_oof[h], global_lgb_test[h] = repeated_binary_model(
        X_global_train, X_global_test, y_map[h], mask_map[h],
        make_lgb_builder(global_lgb_cfgs[h]), TREE_SEEDS, n_splits=5, horizon=h, use_ipcw=(h in [24, 48])
    )
    global_lgb_oof[h] = stabilize(global_lgb_oof[h], anchor_oof[h], 0.92)
    global_lgb_test[h] = stabilize(global_lgb_test[h], anchor_test[h], 0.92)
    print(f"global_lgb@{h}  Brier = {compute_brier(time_values, event_values, global_lgb_oof[h], h):.6f}")

    global_cat_oof[h], global_cat_test[h] = repeated_binary_model(
        X_global_train, X_global_test, y_map[h], mask_map[h],
        make_cat_builder(global_cat_cfgs[h]), CAT_SEEDS, n_splits=5, horizon=h, use_ipcw=(h in [24, 48])
    )
    global_cat_oof[h] = stabilize(global_cat_oof[h], anchor_oof[h], 0.90)
    global_cat_test[h] = stabilize(global_cat_test[h], anchor_test[h], 0.90)
    print(f"global_cat@{h}  Brier = {compute_brier(time_values, event_values, global_cat_oof[h], h):.6f}")

    linear_oof[h], linear_test[h] = repeated_binary_model(
        X_linear_train, X_linear_test, y_map[h], mask_map[h],
        make_logit_builder(linear_cs[h]), LINEAR_SEEDS, n_splits=5, horizon=h, use_ipcw=(h in [24, 48])
    )
    linear_oof[h] = stabilize(linear_oof[h], anchor_oof[h], 0.88)
    linear_test[h] = stabilize(linear_test[h], anchor_test[h], 0.88)
    print(f"linear@{h}      Brier = {compute_brier(time_values, event_values, linear_oof[h], h):.6f}")

near_lgb_oof, near_lgb_test = {}, {}
for h in [12, 24, 48]:
    near_lgb_oof[h], near_lgb_test[h] = repeated_binary_model(
        X_near_lgb_train, X_near_lgb_test, y_map[h], mask_map[h],
        make_lgb_builder(near_lgb_cfgs[h]), TREE_SEEDS, n_splits=5, horizon=h, use_ipcw=(h in [24, 48])
    )
    near_lgb_oof[h] = stabilize(near_lgb_oof[h], anchor_oof[h], 0.94)
    near_lgb_test[h] = stabilize(near_lgb_test[h], anchor_test[h], 0.94)
    print(f"near_lgb@{h}    Brier = {compute_brier(time_values, event_values, near_lgb_oof[h], h):.6f}")

far_lgb_oof, far_lgb_test = {}, {}
for h in [24, 48]:
    far_lgb_oof[h], far_lgb_test[h] = repeated_binary_model(
        X_far_lgb_train, X_far_lgb_test, y_map[h], mask_map[h],
        make_lgb_builder(far_lgb_cfgs[h]), TREE_SEEDS, n_splits=5, horizon=h, use_ipcw=True
    )
    far_lgb_oof[h] = stabilize(far_lgb_oof[h], anchor_oof[h], 0.94)
    far_lgb_test[h] = stabilize(far_lgb_test[h], anchor_test[h], 0.94)
    print(f"far_lgb@{h}     Brier = {compute_brier(time_values, event_values, far_lgb_oof[h], h):.6f}")

timing_signal_train = 1.0 / (np.asarray(train_proc["eta_effective"], dtype=float) / 24.0 + 1.0)
timing_signal_test = 1.0 / (np.asarray(test_proc["eta_effective"], dtype=float) / 24.0 + 1.0)
timing_oof, timing_test = {}, {}
for h in [12, 24, 48]:
    timing_mask = mask_map[h] & ((dist_train < 8000) | (y_map[h] == 1))
    timing_oof[h], timing_test[h] = repeated_isotonic(
        timing_signal_train, timing_signal_test, y_map[h], timing_mask, ANCHOR_SEEDS, n_splits=5
    )
    timing_oof[h] = stabilize(timing_oof[h], anchor_oof[h], 0.90)
    timing_test[h] = stabilize(timing_test[h], anchor_test[h], 0.90)
    print(f"timing@{h}      Brier = {compute_brier(time_values, event_values, timing_oof[h], h):.6f}")

if HAS_XGB:
    print("[INFO] Training native XGBoost survival models ...")
    aft_lower = time_values.astype(float)
    aft_upper = np.where(event_values == 1, time_values.astype(float), np.inf)
    xgb_aft_raw_oof, xgb_aft_raw_test = repeated_xgb_aft_models(
        X_xgb_surv_train, X_xgb_surv_test, aft_lower, aft_upper, XGB_AFT_CFGS, XGB_SURV_SEEDS, n_splits=5
    )
    xgb_aft_oof, xgb_aft_test = {}, {}
    for j, h in enumerate([12, 24, 48]):
        cal_oof, cal_test = repeated_isotonic(
            xgb_aft_raw_oof[:, j], xgb_aft_raw_test[:, j], y_map[h], mask_map[h], ANCHOR_SEEDS, n_splits=5
        )
        alpha_tr = np.clip(0.15 + 0.80 * row_rel_train, 0.35, 0.96)
        alpha_te = np.clip(0.15 + 0.80 * row_rel_test, 0.35, 0.96)
        xgb_aft_oof[h] = stabilize_rowwise(cal_oof, anchor_oof[h], alpha_tr)
        xgb_aft_test[h] = stabilize_rowwise(cal_test, anchor_test[h], alpha_te)
        print(f"xgb_aft@{h}     Brier = {compute_brier(time_values, event_values, xgb_aft_oof[h], h):.6f}")

    cox_label = np.where(event_values == 1, time_values.astype(float), -time_values.astype(float))
    xgb_cox_signal_oof, xgb_cox_signal_test = repeated_xgb_cox_signal(
        X_xgb_surv_train, X_xgb_surv_test, cox_label, XGB_COX_CFGS, XGB_SURV_SEEDS, n_splits=5
    )
    xgb_cox_oof, xgb_cox_test = {}, {}
    for h in [12, 24, 48]:
        cal_oof, cal_test = repeated_isotonic(
            xgb_cox_signal_oof, xgb_cox_signal_test, y_map[h], mask_map[h], ANCHOR_SEEDS, n_splits=5
        )
        alpha_tr = np.clip(0.12 + 0.78 * row_rel_train, 0.30, 0.95)
        alpha_te = np.clip(0.12 + 0.78 * row_rel_test, 0.30, 0.95)
        xgb_cox_oof[h] = stabilize_rowwise(cal_oof, anchor_oof[h], alpha_tr)
        xgb_cox_test[h] = stabilize_rowwise(cal_test, anchor_test[h], alpha_te)
        print(f"xgb_cox@{h}     Brier = {compute_brier(time_values, event_values, xgb_cox_oof[h], h):.6f}")
else:
    print("[WARN] xgboost unavailable: falling back to anchor for native survival channels.")
    xgb_aft_oof = {h: anchor_oof[h].copy() for h in [12, 24, 48]}
    xgb_aft_test = {h: anchor_test[h].copy() for h in [12, 24, 48]}
    xgb_cox_oof = {h: anchor_oof[h].copy() for h in [12, 24, 48]}
    xgb_cox_test = {h: anchor_test[h].copy() for h in [12, 24, 48]}

base_oof = {
    "anchor": anchor_oof,
    "glob_lgb": global_lgb_oof,
    "glob_cat": global_cat_oof,
    "lin": linear_oof,
    "timing": timing_oof,
    "xgb_aft": xgb_aft_oof,
    "xgb_cox": xgb_cox_oof,
}
base_test = {
    "anchor": anchor_test,
    "glob_lgb": global_lgb_test,
    "glob_cat": global_cat_test,
    "lin": linear_test,
    "timing": timing_test,
    "xgb_aft": xgb_aft_test,
    "xgb_cox": xgb_cox_test,
}

near_lgb_store = {"oof": near_lgb_oof, "test": near_lgb_test}
far_lgb_store = {"oof": far_lgb_oof, "test": far_lgb_test}

PRIORS = {
    (12, "near"): {"xgb_aft": 0.28, "xgb_cox": 0.16, "zone_lgb": 0.18, "timing": 0.08, "glob_lgb": 0.10, "glob_cat": 0.07, "lin": 0.05, "anchor": 0.08},
    (12, "far"):  {"xgb_aft": 0.32, "xgb_cox": 0.18, "glob_lgb": 0.14, "glob_cat": 0.10, "lin": 0.08, "timing": 0.08, "anchor": 0.10},
    (24, "near"): {"xgb_aft": 0.26, "xgb_cox": 0.18, "zone_lgb": 0.15, "timing": 0.10, "glob_lgb": 0.10, "glob_cat": 0.08, "lin": 0.04, "anchor": 0.09},
    (24, "far"):  {"xgb_aft": 0.24, "xgb_cox": 0.22, "zone_lgb": 0.16, "glob_lgb": 0.12, "glob_cat": 0.09, "lin": 0.05, "timing": 0.05, "anchor": 0.07},
    (48, "near"): {"xgb_aft": 0.20, "xgb_cox": 0.18, "zone_lgb": 0.18, "timing": 0.10, "glob_lgb": 0.12, "glob_cat": 0.08, "lin": 0.05, "anchor": 0.09},
    (48, "far"):  {"xgb_aft": 0.18, "xgb_cox": 0.22, "zone_lgb": 0.22, "glob_lgb": 0.15, "glob_cat": 0.10, "lin": 0.05, "timing": 0.03, "anchor": 0.05},
}
BLEND_CFG = {
    (12, "near"): {"shrink": 0.30, "reg": 0.12},
    (12, "far"):  {"shrink": 0.35, "reg": 0.10},
    (24, "near"): {"shrink": 0.30, "reg": 0.10},
    (24, "far"):  {"shrink": 0.40, "reg": 0.08},
    (48, "near"): {"shrink": 0.35, "reg": 0.08},
    (48, "far"):  {"shrink": 0.45, "reg": 0.06},
}
GATE_CFG = {
    12: (4500.0, 1200.0),
    24: (5000.0, 1500.0),
    48: (6000.0, 1800.0),
}

def prior_to_vector(prior_dict, names):
    w = np.array([prior_dict.get(n, 0.0) for n in names], dtype=float)
    w = np.clip(w, 0, None)
    if w.sum() <= 0:
        w = np.ones(len(names), dtype=float)
    return w / w.sum()

def assemble_available(h, zone, split_kind, prior_dict):
    store = base_oof if split_kind == "oof" else base_test
    avail = {}
    for key in ["xgb_aft", "xgb_cox", "glob_lgb", "glob_cat", "zone_lgb", "lin", "anchor", "timing"]:
        if key == "zone_lgb":
            src = near_lgb_store[split_kind] if zone == "near" else far_lgb_store[split_kind]
            if h in src:
                avail[key] = src[h]
        else:
            if key in store and h in store[key]:
                avail[key] = store[key][h]
    names = [n for n in prior_dict if n in avail]
    if not names:
        names = list(avail.keys())
    P = np.column_stack([avail[n] for n in names])
    prior = prior_to_vector(prior_dict, names)
    return names, P, prior

def optimize_blend(P, y, prior, reg, shrink):
    if P.shape[1] == 1:
        return prior.copy(), prior.copy()
    P = np.clip(P, 1e-6, 1 - 1e-6)
    y = np.asarray(y, dtype=float)

    def objective(w):
        pred = P @ w
        return np.mean((pred - y) ** 2) + reg * np.sum((w - prior) ** 2)

    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    bounds = [(0.0, 1.0)] * P.shape[1]
    try:
        res = minimize(
            objective, prior.copy(), method="SLSQP", bounds=bounds, constraints=constraints,
            options={"maxiter": 500, "ftol": 1e-10}
        )
        if res.success:
            opt = np.clip(res.x, 0, None)
            opt = opt / opt.sum()
        else:
            opt = prior.copy()
    except Exception:
        opt = prior.copy()

    final = shrink * opt + (1.0 - shrink) * prior
    final = np.clip(final, 0, None)
    final = final / final.sum()
    return final, opt

weights_log = {}
oof_final = np.zeros((len(train_df), 4), dtype=float)
test_final = np.zeros((len(test_df), 4), dtype=float)

for col_idx, h in enumerate([12, 24, 48]):
    center, width = GATE_CFG[h]
    gate_train = soft_gate(dist_train, center, width)
    gate_test = soft_gate(dist_test, center, width)

    prior_near = PRIORS[(h, "near")]
    names_n, Pn_oof, prior_vec_n = assemble_available(h, "near", "oof", prior_near)
    _, Pn_test, _ = assemble_available(h, "near", "test", prior_near)
    near_mask = mask_map[h] & (dist_train < 5000)
    if near_mask.sum() >= max(8, len(names_n) + 2):
        w_n, opt_n = optimize_blend(
            Pn_oof[near_mask], y_map[h][near_mask], prior_vec_n,
            BLEND_CFG[(h, "near")]["reg"], BLEND_CFG[(h, "near")]["shrink"]
        )
    else:
        w_n, opt_n = prior_vec_n.copy(), prior_vec_n.copy()
    pred_near_oof = np.clip(Pn_oof @ w_n, 0, 1)
    pred_near_test = np.clip(Pn_test @ w_n, 0, 1)

    prior_far = PRIORS[(h, "far")]
    names_f, Pf_oof, prior_vec_f = assemble_available(h, "far", "oof", prior_far)
    _, Pf_test, _ = assemble_available(h, "far", "test", prior_far)
    far_mask = mask_map[h] & ~(dist_train < 5000)
    if far_mask.sum() >= max(8, len(names_f) + 2):
        w_f, opt_f = optimize_blend(
            Pf_oof[far_mask], y_map[h][far_mask], prior_vec_f,
            BLEND_CFG[(h, "far")]["reg"], BLEND_CFG[(h, "far")]["shrink"]
        )
    else:
        w_f, opt_f = prior_vec_f.copy(), prior_vec_f.copy()
    pred_far_oof = np.clip(Pf_oof @ w_f, 0, 1)
    pred_far_test = np.clip(Pf_test @ w_f, 0, 1)

    oof_final[:, col_idx] = gate_train * pred_near_oof + (1.0 - gate_train) * pred_far_oof
    test_final[:, col_idx] = gate_test * pred_near_test + (1.0 - gate_test) * pred_far_test

    alpha_tr = np.clip(0.25 + 0.70 * row_rel_train, 0.38, 0.97)
    alpha_te = np.clip(0.25 + 0.70 * row_rel_test, 0.38, 0.97)
    oof_final[:, col_idx] = stabilize_rowwise(oof_final[:, col_idx], anchor_oof[h], alpha_tr)
    test_final[:, col_idx] = stabilize_rowwise(test_final[:, col_idx], anchor_test[h], alpha_te)

    weights_log[f"{h}_near"] = {
        "names": names_n,
        "prior": [float(x) for x in prior_vec_n],
        "opt": [float(x) for x in opt_n],
        "final": [float(x) for x in w_n],
    }
    weights_log[f"{h}_far"] = {
        "names": names_f,
        "prior": [float(x) for x in prior_vec_f],
        "opt": [float(x) for x in opt_f],
        "final": [float(x) for x in w_f],
    }

oof_final[:, 3] = 1.0
test_final[:, 3] = 1.0
oof_final = enforce_monotonicity(oof_final)
test_final = enforce_monotonicity(test_final)

if DO_OOF:
    hybrid, c_idx, wb, b24, b48, b72 = compute_hybrid_score(
        time_values, event_values, oof_final[:, 1], oof_final[:, 2], oof_final[:, 3]
    )
    b12 = compute_brier(time_values, event_values, oof_final[:, 0], 12)
    print(f"OOF Hybrid = {hybrid:.6f}")
    print(f"OOF C-Index = {c_idx:.6f}")
    print(f"OOF WBrier = {wb:.6f}")
    print(f"OOF Brier@12 = {b12:.6f}")
    print(f"OOF Brier@24 = {b24:.6f}")
    print(f"OOF Brier@48 = {b48:.6f}")
    print(f"OOF Brier@72 = {b72:.6f}")

submission = pd.DataFrame({
    "event_id": test_df["event_id"].values,
    "prob_12h": test_final[:, 0],
    "prob_24h": test_final[:, 1],
    "prob_48h": test_final[:, 2],
    "prob_72h": test_final[:, 3],
})
submission = sample_sub[["event_id"]].merge(submission, on="event_id", how="left", validate="one_to_one")

def validate_submission(sub, sample):
    required = ["event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"]
    assert list(sub.columns) == required
    assert len(sub) == len(sample)
    assert sub["event_id"].equals(sample["event_id"])
    assert sub["event_id"].nunique() == len(sub)
    vals = sub[required[1:]].values
    assert np.isfinite(vals).all()
    assert ((vals >= 0) & (vals <= 1)).all()
    assert np.all(vals[:, 0] <= vals[:, 1] + 1e-12)
    assert np.all(vals[:, 1] <= vals[:, 2] + 1e-12)
    assert np.all(vals[:, 2] <= vals[:, 3] + 1e-12)

validate_submission(submission, sample_sub)

os.makedirs(WORK_DIR, exist_ok=True)
submission.to_csv(OUTPUT_PATH, index=False)

if DO_OOF:
    oof_dump = pd.DataFrame({
        "event_id": train_df["event_id"].values,
        "prob_12h": oof_final[:, 0],
        "prob_24h": oof_final[:, 1],
        "prob_48h": oof_final[:, 2],
        "prob_72h": oof_final[:, 3],
        "event": event_values,
        "time_to_hit_hours": time_values,
    })
    oof_dump.to_csv(os.path.join(WORK_DIR, "oof_predictions.csv"), index=False)

with open(os.path.join(WORK_DIR, "blend_weights.json"), "w") as f:
    json.dump(weights_log, f, indent=2)

print("Saved:", OUTPUT_PATH)
print(submission.head())

# --- Optional external consensus blend ---
core_submission = submission.copy()
core_path = os.path.join(WORK_DIR, "submission_core.csv")
core_submission.to_csv(core_path, index=False)

REQUIRED_COLS = ["event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"]

def _rank_to_core_distribution(core_vals, rank_signal):
    order = np.argsort(rank_signal, kind="mergesort")
    sorted_core = np.sort(core_vals)
    out = np.empty_like(sorted_core)
    out[order] = sorted_core
    return out

def _safe_read_submission_csv(path, sample_ids):
    try:
        df = pd.read_csv(path)
    except Exception:
        return None
    if list(df.columns) != REQUIRED_COLS:
        return None
    if len(df) != len(sample_ids):
        return None
    if df["event_id"].duplicated().any():
        return None
    if not df["event_id"].equals(sample_ids):
        try:
            df = sample_ids.to_frame(name="event_id").merge(df, on="event_id", how="left", validate="one_to_one")
        except Exception:
            return None
        if df.isnull().any().any():
            return None
    vals = df[REQUIRED_COLS[1:]].to_numpy(dtype=float)
    if not np.isfinite(vals).all():
        return None
    vals = np.clip(vals, 0.0, 1.0)
    vals = enforce_monotonicity(vals)
    vals[:, 3] = np.maximum(vals[:, 3], vals[:, 2])
    df.loc[:, REQUIRED_COLS[1:]] = vals
    return df

def _extract_score_from_name(path):
    name = os.path.basename(path)
    m = re.findall(r"(\d\.\d{4,5})", name)
    if not m:
        return None
    try:
        return max(float(x) for x in m)
    except Exception:
        return None

def _score_from_linked_notebook(path):
    name = os.path.basename(path)
    m = re.match(r"samplecsv_sub(\d+)\.csv$", name)
    if not m:
        return _extract_score_from_name(path)
    idx = m.group(1)
    patt = re.compile(rf"final_sub{idx}_(\d\.\d{{4,5}})\.ipynb$")
    roots = []
    for p in [os.path.dirname(path), DATA_DIR, "/kaggle/input", "/kaggle/working", ".", "/mnt/data"]:
        if p and os.path.isdir(p):
            roots.append(p)
    best = None
    for root in roots:
        for cur, _, files in os.walk(root):
            for fn in files:
                mm = patt.match(fn)
                if mm:
                    try:
                        score = float(mm.group(1))
                        best = score if best is None else max(best, score)
                    except Exception:
                        pass
    if best is not None:
        return best
    return _extract_score_from_name(path)

def _hash_frame(df):
    arr = np.round(df[REQUIRED_COLS[1:]].to_numpy(dtype=float), 8)
    return hashlib.md5(arr.tobytes()).hexdigest()

def find_candidate_submission_paths():
    roots = []
    for p in [DATA_DIR, "/kaggle/input", "/kaggle/working", ".", "/mnt/data"]:
        if p and os.path.isdir(p):
            roots.append(p)
    deny_names = {
        "train.csv", "test.csv", "sample_submission.csv",
        "submission.csv", "submission_core.csv", "submission_external.csv",
        "oof_predictions.csv"
    }
    paths = []
    seen = set()
    for root in roots:
        for cur, _, files in os.walk(root):
            for fn in files:
                low = fn.lower()
                if not low.endswith(".csv"):
                    continue
                if fn in deny_names:
                    continue
                full = os.path.join(cur, fn)
                if full in seen:
                    continue
                seen.add(full)
                paths.append(full)
    return sorted(paths)

sample_ids = sample_sub["event_id"]
candidate_paths = find_candidate_submission_paths()
candidate_frames = []
candidate_meta = []

for path in candidate_paths:
    cand = _safe_read_submission_csv(path, sample_ids)
    if cand is None:
        continue
    if np.allclose(cand[REQUIRED_COLS[1:]].to_numpy(), core_submission[REQUIRED_COLS[1:]].to_numpy(), atol=1e-12):
        continue
    score = _score_from_linked_notebook(path)
    arr = cand[REQUIRED_COLS[1:]].to_numpy(dtype=float)
    core_arr = core_submission[REQUIRED_COLS[1:]].to_numpy(dtype=float)
    corrs = []
    for j in range(3):
        x = arr[:, j]
        y = core_arr[:, j]
        if np.std(x) < 1e-12 or np.std(y) < 1e-12:
            corrs.append(1.0)
        else:
            corrs.append(float(np.corrcoef(x, y)[0, 1]))
    mean_corr = float(np.mean(corrs))
    if not np.isfinite(mean_corr) or mean_corr < 0.70:
        continue
    candidate_frames.append(cand)
    candidate_meta.append({
        "path": path,
        "score": score,
        "mean_corr_12_48": mean_corr,
        "hash": _hash_frame(cand),
    })

dedup_frames = []
dedup_meta = []
seen_hash = set()
for df, meta in zip(candidate_frames, candidate_meta):
    if meta["hash"] in seen_hash:
        continue
    seen_hash.add(meta["hash"])
    dedup_frames.append(df)
    dedup_meta.append(meta)
candidate_frames = dedup_frames
candidate_meta = dedup_meta

if candidate_frames:
    raw_weights = []
    for meta in candidate_meta:
        s = meta["score"]
        c = meta["mean_corr_12_48"]
        if s is None:
            w = 1.0
        else:
            w = max(0.001, s - 0.94) / 0.04
        w *= max(0.10, c)
        raw_weights.append(w)
    raw_weights = np.asarray(raw_weights, dtype=float)
    weights = raw_weights / raw_weights.sum()

    candidate_arrays = np.stack([df[REQUIRED_COLS[1:]].to_numpy(dtype=float) for df in candidate_frames], axis=0)
    prob_mean = np.tensordot(weights, candidate_arrays, axes=(0, 0))

    rankblend = np.zeros_like(prob_mean)
    core_vals = core_submission[REQUIRED_COLS[1:]].to_numpy(dtype=float)
    for j in range(4):
        rank_signal = np.tensordot(
            weights,
            np.stack([df[REQUIRED_COLS[1:]].rank(method="average", pct=True).to_numpy(dtype=float)[:, j] for df in candidate_frames], axis=0),
            axes=(0, 0)
        )
        rankblend[:, j] = _rank_to_core_distribution(core_vals[:, j], rank_signal)

    final_vals = core_vals.copy()
    if len(candidate_meta) >= 5:
        prob_w = np.array([0.03, 0.06, 0.12, 0.00], dtype=float)
        rank_w = np.array([0.04, 0.06, 0.10, 0.00], dtype=float)
    else:
        prob_w = np.array([0.02, 0.05, 0.10, 0.00], dtype=float)
        rank_w = np.array([0.03, 0.05, 0.08, 0.00], dtype=float)

    for j in range(4):
        keep_w = 1.0 - prob_w[j] - rank_w[j]
        final_vals[:, j] = keep_w * core_vals[:, j] + prob_w[j] * prob_mean[:, j] + rank_w[j] * rankblend[:, j]

    final_vals = np.clip(final_vals, 0.0, 1.0)
    final_vals = enforce_monotonicity(final_vals)
    final_vals[:, 3] = 1.0

    external_submission = core_submission.copy()
    external_submission.loc[:, REQUIRED_COLS[1:]] = final_vals
    validate_submission(external_submission, sample_sub)

    external_path = os.path.join(WORK_DIR, "submission_external.csv")
    external_submission.to_csv(external_path, index=False)
    external_submission.to_csv(OUTPUT_PATH, index=False)

    with open(os.path.join(WORK_DIR, "external_blend_summary.json"), "w") as f:
        json.dump({
            "n_candidates": len(candidate_meta),
            "weights": [
                {
                    "path": m["path"],
                    "score": m["score"],
                    "mean_corr_12_48": m["mean_corr_12_48"],
                    "weight": float(w),
                }
                for m, w in zip(candidate_meta, weights)
            ],
            "blend": {
                "prob_w": prob_w.tolist(),
                "rank_w": rank_w.tolist(),
            }
        }, f, indent=2)

    print("External candidates used:", len(candidate_meta))
    for m, w in sorted(zip(candidate_meta, weights), key=lambda z: -z[1])[:20]:
        print(f"  w={w:.4f} | score={m['score']} | corr={m['mean_corr_12_48']:.4f} | {m['path']}")
    print("Saved:", core_path)
    print("Saved:", external_path)
    print("Saved:", OUTPUT_PATH)
else:
    core_submission.to_csv(os.path.join(WORK_DIR, "submission_external.csv"), index=False)
    core_submission.to_csv(OUTPUT_PATH, index=False)
    with open(os.path.join(WORK_DIR, "external_blend_summary.json"), "w") as f:
        json.dump({"n_candidates": 0, "weights": [], "blend": None}, f, indent=2)
    print("No external CSV candidates found. submission.csv = submission_core.csv")
    print("Saved:", core_path)
    print("Saved:", os.path.join(WORK_DIR, "submission_external.csv"))
    print("Saved:", OUTPUT_PATH)


DATA_DIR = /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26
TRAIN = (221, 37) TEST = (95, 35) HAS_XGB = True
anchor@12      Brier = 0.068924
anchor@24      Brier = 0.029597
anchor@48      Brier = 0.019173
global_lgb@12  Brier = 0.059285
global_cat@12  Brier = 0.054787
linear@12      Brier = 0.056295
global_lgb@24  Brier = 0.027339
global_cat@24  Brier = 0.026114
linear@24      Brier = 0.030228
global_lgb@48  Brier = 0.016093
global_cat@48  Brier = 0.015946
linear@48      Brier = 0.017943
near_lgb@12    Brier = 0.057101
near_lgb@24    Brier = 0.027013
near_lgb@48    Brier = 0.015905
far_lgb@24     Brier = 0.027485
far_lgb@48     Brier = 0.015582
timing@12      Brier = 0.211354
timing@24      Brier = 0.340425
timing@48      Brier = 0.368167
[INFO] Training native XGBoost survival models ...
xgb_aft@12     Brier = 0.060767
xgb_aft@24     Brier = 0.034110
xgb_aft@48     Brier = 0.022822
xgb_cox@12     Brier = 0.059823
xgb_cox@24     Brier = 0.033218
xgb_cox@48     Brier = 0.022213
O